In [ ]:
!pip install langchain_community
!pip install crewai crewai_tools

In [ ]:
from google.colab import userdata
import os

os.environ['GEMINI_API_KEY']=userdata.get('GOOGLE_API_KEY')
os.environ['TAVILY_API_KEY']=userdata.get('TAVILY_API_KEY')

In [ ]:
import os
from crewai import Agent, Task, Crew
from crewai.tools import BaseTool
from pydantic import Field
from langchain_community.tools import TavilySearchResults


# Initialize the LangChain Tavily tool
tavily_search = TavilySearchResults()

# Define the custom tool for Tavily search
class TavilySearchTool(BaseTool):
    name: str = "Tavily Search"
    description: str = (
        "Useful for retrieving structured search results from Tavily. "
        "Use this tool to find information on various topics efficiently."
    )
    tavily_search: TavilySearchResults = Field(default_factory=TavilySearchResults)

    def _run(self, query: str) -> str:
        """Execute the Tavily search query and return results."""
        try:
            return self.tavily_search.run(query)
        except Exception as e:
            return f"Error performing Tavily search: {str(e)}"

In [ ]:
# Create an agent and assign the TavilySearchTool
market_researcher = Agent(
    role='Market Researcher',
    goal='Provide insights on current market trends using structured search results.',
    backstory=(
        "You are a skilled market researcher with expertise in gathering and analyzing "
        "data to identify trends and insights for strategic decision-making."
    ),
    llm="gemini/gemini-2.0-flash-exp",
    tools=[TavilySearchTool()],
    verbose=True
)

# Create a CrewAI Task
task = Task(
    description="Identify trends in the technology sector for the last quarter.",
    expected_output="Description of market trends and insights formatted as markdown.",
    agent=market_researcher,
    output_file="marketing_report.md"
)

# Create the Crew and run the task
crew = Crew(agents=[market_researcher],
            tasks=[task],
            verbose=True)

crew.kickoff()